In [7]:
import yaml
from ssm import *

model = SSMTranslator(config=SSMTranslatorConfig(**yaml.safe_load(open("../configs/prescale_2/ssm_model_config_1.yaml", "r"))))
state_dict = { x[len("module."):]: v for x, v in torch.load("../temp/best_model.pt").items() }
model.load_state_dict(state_dict)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SSMTranslator(
  (E): Embedding(32000, 64)
  (encoder_layers): ModuleList(
    (0-1): 2 x Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(320, 320, kernel_size=(4,), stride=(1,), groups=320)
      (lin_gate): MultiheadLinear()
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (lin_out): Linear(in_features=64, out_features=64, bias=True)
    )
    (2): Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(192, 192, kernel_size=(4,), stride=(1,), groups=192)
    )
  )
  (lin_EDs): ModuleList(
    (0-2): 3 x Linear(in_features=32, out_features=16, bias=True)
  )
  (decoder_layers): ModuleList(
    (0-2): 3 x Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(192, 192, kernel_size=(4,), stride=(1,), groups=192)
      (lin_gate): MultiheadLinear()
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=T

In [20]:
import sentencepiece as sp

proc_en = sp.SentencePieceProcessor()
proc_fr = sp.SentencePieceProcessor()

proc_en.Load("../vocab/en.model")
proc_fr.Load("../vocab/fr.model")

sentence = "Bonjour, madame. Comment allez-vous aujourd'hui?"
en_ids = proc_en.Encode(sentence, add_bos=True, add_eos=True)
en_ids = torch.tensor(en_ids).unsqueeze(0).to(device)

fr_ids = proc_fr.Encode("Bonjour, comment ça va aujourd'hui?", add_bos=True, add_eos=True)
fr_ids = torch.tensor(fr_ids).unsqueeze(0).to(device)

print(en_ids)
print(fr_ids)

logits = model(
    inp_ids=en_ids,
    decode_method="ag",
    max_output_len=50,
    # forcing_ids=fr_ids[:, :-1],
)

ids = torch.argmax(logits, dim=-1).squeeze(0).tolist()
print(ids)

print(proc_fr.Decode(ids))

tensor([[    1, 10544, 30348, 31037,    43,    61,   588, 31036, 11612,  9608,
         31077, 31046, 29850,  7528, 30348, 31025, 31080, 12065, 31018, 31095,
             2]], device='cuda:0')
tensor([[    1,  9365,  2090, 31000,  1857, 13122,  4750,  3270, 31004,  2314,
         31076,     2]], device='cuda:0')
[2989, 5756, 1878, 31, 795, 101, 38, 919, 11, 26, 616, 101, 26, 1109, 11, 26, 809, 57, 1864, 11, 26, 1076, 31001, 2]
Vous devez fournir des renseignements sur les résultats de la recherche sur la base de la partie du poste de la personne.
